# Kenshi hazard model: fine-tune on Colab

Fine-tunes YOLO26n (`yolo26n.pt`, 80 COCO classes) into an 85-class model that adds **stairs, curb, pole, door, desk**
(ids 80 to 84), at the 416 px input size chosen at Gate A, then exports a fixed-size ONNX file.

**Before you start, on your own computer:** run `uv run --group ml python ml/prepare_dataset.py --zip`, then upload
`datasets/hazards.zip` to Google Drive as `My Drive/kenshi/hazards.zip`.

**In Colab:** Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. A free T4 needs from about one to a few
hours for 60 epochs, depending on the dataset size. Free sessions can disconnect; progress is saved to Drive after every
epoch, so if that happens just run all again and training resumes.

Cells marked **(Colab only)** cannot run on a normal computer: they use Google Drive, `/content`, or the Colab GPU.

## 1. Connect to Google Drive (Colab only)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Settings

In [ ]:
from pathlib import Path

DRIVE = Path(
    "/content/drive/MyDrive/kenshi"
)  # where hazards.zip is, and where results are saved
IMG_SIZE = 416  # the input size chosen at Gate A
EPOCHS = 60
PATIENCE = 15  # stop early if validation does not improve for this many epochs
BATCH = 32
BASE_WEIGHTS = "yolo26n.pt"
RUN_NAME = f"hazards-{IMG_SIZE}"
RUNS = DRIVE / "runs"  # on Drive, so a disconnect does not lose progress

assert (DRIVE / "hazards.zip").exists(), "Upload hazards.zip to My Drive/kenshi first"
print("found", DRIVE / "hazards.zip")

## 3. Install and check the GPU (Colab only)

In [ ]:
# Same Ultralytics version as the project's uv.lock, so the export matches the app.
%pip install -q "ultralytics==8.4.175" onnx onnxslim onnxruntime

import torch

assert torch.cuda.is_available(), (
    "No GPU: Runtime > Change runtime type > T4 GPU, then run again"
)
print("GPU:", torch.cuda.get_device_name(0))

## 4. Unpack the dataset onto Colab's fast local disk (Colab only)

In [ ]:
import zipfile

import yaml

LOCAL = Path("/content/datasets")
if not (LOCAL / "hazards").exists():
    LOCAL.mkdir(parents=True, exist_ok=True)
    print("unpacking ...")
    with zipfile.ZipFile(DRIVE / "hazards.zip") as z:
        z.extractall(LOCAL)

DATA_YAML = LOCAL / "hazards" / "data.yaml"
data = yaml.safe_load(DATA_YAML.read_text())
data["path"] = str(LOCAL / "hazards")  # the zip was made on another computer
DATA_YAML.write_text(yaml.safe_dump(data, sort_keys=False))

train_images = len(list((LOCAL / "hazards" / "images" / "train").iterdir()))
val_images = len(list((LOCAL / "hazards" / "images" / "val").iterdir()))
print(
    len(data["names"]),
    "classes;",
    train_images,
    "training images;",
    val_images,
    "validation images",
)
assert len(data["names"]) == 85, (
    "expected 85 classes (80 COCO + stairs, curb, pole, door, desk)"
)

## 5. Train

Starts from `yolo26n.pt`. Because the class count changes from 80 to 85, the final classification layers start fresh
and are learned from our data; the rest of the network keeps its pretrained weights. If a run already exists on Drive
this resumes it.

In [ ]:
from ultralytics import YOLO

last = RUNS / RUN_NAME / "weights" / "last.pt"
if last.exists():
    print("resuming from", last)
    try:
        YOLO(last).train(resume=True)
    except AssertionError as error:
        print("nothing to resume (the run already finished):", error)
else:
    model = YOLO(BASE_WEIGHTS)
    model.train(
        data=str(DATA_YAML),
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        patience=PATIENCE,
        batch=BATCH,
        workers=2,
        project=str(RUNS),
        name=RUN_NAME,
        exist_ok=True,
        seed=0,
        close_mosaic=10,
        # The app sees objects cut off by the frame edge; shift and zoom more than the defaults
        # (0.1 and 0.5) so the model sees many more of them.
        translate=0.25,
        scale=0.6,
    )

## 6. Validation mAP per class

In [ ]:
best_path = RUNS / RUN_NAME / "weights" / "best.pt"
best = YOLO(best_path)
metrics = best.val(data=str(DATA_YAML), imgsz=IMG_SIZE, plots=False)

print(f"{'class':18s} {'mAP50':>7} {'mAP50-95':>9}")
for i, class_id in enumerate(metrics.box.ap_class_index):
    print(
        f"{best.names[int(class_id)]:18s} {metrics.box.ap50[i]:7.3f} {metrics.box.ap[i]:9.3f}"
    )
print(f"{'all classes':18s} {metrics.box.map50:7.3f} {metrics.box.map:9.3f}")

## 7. Export ONNX at the same fixed size

Same settings as `ml/export_onnx.py`, so the output has the layout the app expects: input `[1, 3, 416, 416]`, output
`[1, 4 + 85, N]` (raw boxes and scores, no NMS in the graph).

In [ ]:
import shutil

import onnx

exported = Path(
    best.export(format="onnx", imgsz=IMG_SIZE, simplify=True, dynamic=False)
)
target = DRIVE / f"yolo26n-hazards-{IMG_SIZE}.onnx"
shutil.copy(exported, target)

graph = onnx.load(target).graph
dims = lambda tensor: [d.dim_value for d in tensor.type.tensor_type.shape.dim]
size_mb = target.stat().st_size / 1e6
print(
    target.name,
    f"{size_mb:.1f} MB",
    "input",
    dims(graph.input[0]),
    "output",
    dims(graph.output[0]),
)
assert size_mb < 15, "the app limits each ONNX file to 15 MB"
assert dims(graph.output[0])[1] == 4 + 85, "unexpected output layout"

## 8. Download the results

In Google Drive, `My Drive/kenshi/` now has:

- `yolo26n-hazards-416.onnx`: the model to compare. Put it on your computer, for example at `datasets/yolo26n-hazards-416.onnx`.
- `runs/hazards-416/`: weights (`weights/best.pt`), training curves, and a confusion matrix.

Then run, on your computer: `uv run --group ml python ml/compare_models.py --hazard datasets/yolo26n-hazards-416.onnx`

Do not commit the weights or the ONNX file; step M3 decides whether the new model replaces the base model.